# Max Chelminski

## HW6 SQL and Scraping

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np
import math
pd.set_option('display.max_colwidth', None)

## 1. **Web-scrape all table data from the following webpage and build a pd.DataFrame object. Limit your final table to include columns for "Package", "Item", "Title", "Rows", and "Cols". Print the dimensions and first five rows of the table.**

In [2]:
url = 'http://vincentarelbundock.github.io/Rdatasets/datasets.html'
response = requests.get(url)
print(response)

<Response [200]>


In [3]:
soup = BeautifulSoup(response.content, 'html.parser')
main_table = soup.find_all('table')[0]

In [4]:
df = pd.read_html(str(main_table))[0]
df

C:\Users\Max\AppData\Local\Temp\ipykernel_22512\1578651500.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(main_table))[0]


,Package,Item,Title,Rows,Cols,n_binary,n_character,n_factor,n_logical,n_numeric,CSV,Doc
0,AER,Affairs,Fair's Extramarital Affairs Data,601.0,9.0,2.0,0.0,2.0,0.0,7.0,CSV,DOC
1,AER,ArgentinaCPI,Consumer Price Index in Argentina,80.0,2.0,0.0,0.0,0.0,0.0,2.0,CSV,DOC
2,AER,BankWages,Bank Wages,474.0,4.0,2.0,0.0,3.0,0.0,1.0,CSV,DOC
3,AER,BenderlyZwick,"Benderly and Zwick Data: Inflation, Growth and Stock Returns",31.0,5.0,0.0,0.0,0.0,0.0,5.0,CSV,DOC
4,AER,BondYield,Bond Yield Data,60.0,2.0,0.0,0.0,0.0,0.0,2.0,CSV,DOC
...,...,...,...,...,...,...,...,...,...,...,...,...
3495,wooldridge,wage2,wage2,935.0,17.0,4.0,0.0,0.0,0.0,17.0,CSV,DOC
3496,wooldridge,wagepan,wagepan,4360.0,44.0,37.0,0.0,0.0,0.0,44.0,CSV,DOC
3497,wooldridge,wageprc,wageprc,286.0,20.0,0.0,0.0,0.0,0.0,20.0,CSV,DOC
3498,wooldridge,wine,wine,21.0,5.0,0.0,1.0,0.0,0.0,4.0,CSV,DOC


In [5]:
df_final = df[['Package', 'Item', 'Title', 'Rows', 'Cols']].copy()
print(df_final.shape)

(3500, 5)


In [6]:
df_final.head()

,Package,Item,Title,Rows,Cols
0,AER,Affairs,Fair's Extramarital Affairs Data,601.0,9.0
1,AER,ArgentinaCPI,Consumer Price Index in Argentina,80.0,2.0
2,AER,BankWages,Bank Wages,474.0,4.0
3,AER,BenderlyZwick,"Benderly and Zwick Data: Inflation, Growth and Stock Returns",31.0,5.0
4,AER,BondYield,Bond Yield Data,60.0,2.0


## 2. **Web-scrape the full links to every CSV file listed in the CSV column of the web-page. Add a new column to your data frame that includes these links. Name the column "csv_links". Print the first five rows of the table.**

In [7]:
csv_links = []
for link in soup.find_all('a'):
    href = link.get('href')

    if href.endswith('.csv'):
        csv_links.append(href)
print(csv_links[:5])

['https://vincentarelbundock.github.io/Rdatasets/csv/AER/Affairs.csv', 'https://vincentarelbundock.github.io/Rdatasets/csv/AER/ArgentinaCPI.csv', 'https://vincentarelbundock.github.io/Rdatasets/csv/AER/BankWages.csv', 'https://vincentarelbundock.github.io/Rdatasets/csv/AER/BenderlyZwick.csv', 'https://vincentarelbundock.github.io/Rdatasets/csv/AER/BondYield.csv']


In [8]:
df_final['csv_links'] = pd.Series(csv_links)
df_final.head()

,Package,Item,Title,Rows,Cols,csv_links
0,AER,Affairs,Fair's Extramarital Affairs Data,601.0,9.0,https://vincentarelbundock.github.io/Rdatasets/csv/AER/Affairs.csv
1,AER,ArgentinaCPI,Consumer Price Index in Argentina,80.0,2.0,https://vincentarelbundock.github.io/Rdatasets/csv/AER/ArgentinaCPI.csv
2,AER,BankWages,Bank Wages,474.0,4.0,https://vincentarelbundock.github.io/Rdatasets/csv/AER/BankWages.csv
3,AER,BenderlyZwick,"Benderly and Zwick Data: Inflation, Growth and Stock Returns",31.0,5.0,https://vincentarelbundock.github.io/Rdatasets/csv/AER/BenderlyZwick.csv
4,AER,BondYield,Bond Yield Data,60.0,2.0,https://vincentarelbundock.github.io/Rdatasets/csv/AER/BondYield.csv


## 3. **Search the "Title" column to return the row of data with the title "Violent Crime Rates by US State".**

In [9]:
df_final[df_final['Title'] == 'Violent Crime Rates by US State'] # Looks like there are three rows with this title

,Package,Item,Title,Rows,Cols,csv_links
838,crimedatasets,USArrests_df,Violent Crime Rates by US State,50.0,4.0,https://vincentarelbundock.github.io/Rdatasets/csv/crimedatasets/USArrests_df.csv
1068,datasets,USArrests,Violent Crime Rates by US State,50.0,4.0,https://vincentarelbundock.github.io/Rdatasets/csv/datasets/USArrests.csv
3294,usdatasets,USArrests_df,Violent Crime Rates by US State,50.0,4.0,https://vincentarelbundock.github.io/Rdatasets/csv/usdatasets/USArrests_df.csv


## 4. **Import the csv file for the dataset identified in #3 using the full link listed in the "csv_links" column for this dataset. Create a new variable called "violent_crime" that adds together data for all columns in the dataset that contain violent crime data (i.e.-add data from assault, murder, and rape columns together in new column called "violent_crime").**

In [10]:
df_final[df_final['Title'] == 'Violent Crime Rates by US State']['csv_links']

838     https://vincentarelbundock.github.io/Rdatasets/csv/crimedatasets/USArrests_df.csv
1068            https://vincentarelbundock.github.io/Rdatasets/csv/datasets/USArrests.csv
3294       https://vincentarelbundock.github.io/Rdatasets/csv/usdatasets/USArrests_df.csv
Name: csv_links, dtype: object

In [11]:
csv_link = 'https://vincentarelbundock.github.io/Rdatasets/csv/crimedatasets/USArrests_df.csv'
df_q4 = pd.read_csv(csv_link)
df_q4.head(1)

,rownames,Murder,Assault,UrbanPop,Rape
0,Alabama,13.2,236,58,21.2


In [12]:
df_q4['violent_crime'] = df_q4['Murder'] + df_q4['Assault'] + df_q4['Rape']
df_q4.head()

,rownames,Murder,Assault,UrbanPop,Rape,violent_crime
0,Alabama,13.2,236,58,21.2,270.4
1,Alaska,10.0,263,48,44.5,317.5
2,Arizona,8.1,294,80,31.0,333.1
3,Arkansas,8.8,190,50,19.5,218.3
4,California,9.0,276,91,40.6,325.6


## 5. **Merge all the data from the states.csv data set (found in the data folder of this HW6 directory) with your dataset to your Violent Crime Rates data frame. Print the first five lines of your new dataset.**

In [13]:
path = '../../Data/states.csv'
df_q5 = pd.read_csv(path)
df_q5.head(1)

,state.name,state.abb,state.area,state.division,state.name.1,state.region,Population,Income,Illiteracy,Life.Exp,Murder,HS.Grad,Frost,Area
0,Alabama,AL,51609,4,Alabama,2,3615,3624,2.1,69.05,15.1,41.3,20,50708


In [14]:
df_q5_final = pd.merge(df_q5, df_q4, left_on='state.name', right_on='rownames')
df_q5_final = df_q5_final.drop(columns=['state.name.1', 'rownames'], axis=1)
df_q5_final.head()

,state.name,state.abb,state.area,state.division,state.region,Population,Income,Illiteracy,Life.Exp,Murder_x,HS.Grad,Frost,Area,Murder_y,Assault,UrbanPop,Rape,violent_crime
0,Alabama,AL,51609,4,2,3615,3624,2.1,69.05,15.1,41.3,20,50708,13.2,236,58,21.2,270.4
1,Alaska,AK,589757,9,4,365,6315,1.5,69.31,11.3,66.7,152,566432,10.0,263,48,44.5,317.5
2,Arizona,AZ,113909,8,4,2212,4530,1.8,70.55,7.8,58.1,15,113417,8.1,294,80,31.0,333.1
3,Arkansas,AR,53104,5,2,2110,3378,1.9,70.66,10.1,39.9,65,51945,8.8,190,50,19.5,218.3
4,California,CA,158693,9,4,21198,5114,1.1,71.71,10.3,62.6,20,156361,9.0,276,91,40.6,325.6


## 6. **Calculate the average for each numeric column in the dataset.**

In [15]:
df_q6 = df_q5_final.select_dtypes(include=np.number)
df_q6.head(1)

,state.area,state.division,state.region,Population,Income,Illiteracy,Life.Exp,Murder_x,HS.Grad,Frost,Area,Murder_y,Assault,UrbanPop,Rape,violent_crime
0,51609,4,2,3615,3624,2.1,69.05,15.1,41.3,20,50708,13.2,236,58,21.2,270.4


In [16]:
df_q6_means = df_q6.mean()
df_q6_means

state.area        72367.9800
state.division        5.2000
state.region          2.5800
Population         4246.4200
Income             4435.8000
Illiteracy            1.1700
Life.Exp             70.8786
Murder_x              7.3780
HS.Grad              53.1080
Frost               104.4600
Area              70735.8800
Murder_y              7.7880
Assault             170.7600
UrbanPop             65.5400
Rape                 21.2320
violent_crime       199.7800
dtype: float64

## 7. **Group the data by region and then calculate the average for each numeric column in the dataset per region. Which region had the highest population (data is from the late 1970s)? Which region had the most violent crime?**

In [17]:
df_q7 = df_q6.groupby(['state.region']).mean()
df_q7

,state.area,state.division,Population,Income,Illiteracy,Life.Exp,Murder_x,HS.Grad,Frost,Area,Murder_y,Assault,UrbanPop,Rape,violent_crime
state.region,,,,,,,,,,,,,,,
1,18817.000000,1.333333,5495.111111,4570.222222,1.000000,71.264444,4.722222,53.966667,132.777778,18141.000,4.700000,126.666667,70.555556,13.777778,145.144444
2,56222.250000,3.750000,4208.125000,4011.937500,1.737500,69.706250,10.581250,44.343750,64.625000,54605.125,11.706250,220.000000,59.437500,21.162500,252.868750
3,63794.166667,6.583333,4803.000000,4611.083333,0.700000,71.766667,5.275000,54.516667,138.833333,62652.000,5.700000,120.333333,64.416667,18.441667,144.475000
4,137227.692308,8.384615,2915.307692,4702.615385,1.023077,71.234615,7.215385,62.000000,102.153846,134463.000,7.030769,187.230769,70.615385,29.053846,223.315385


In [18]:
df_q7.sort_values('Population', ascending=False).head(1)

,state.area,state.division,Population,Income,Illiteracy,Life.Exp,Murder_x,HS.Grad,Frost,Area,Murder_y,Assault,UrbanPop,Rape,violent_crime
state.region,,,,,,,,,,,,,,,
1,18817.0,1.333333,5495.111111,4570.222222,1.0,71.264444,4.722222,53.966667,132.777778,18141.0,4.7,126.666667,70.555556,13.777778,145.144444


Region 1 had the highest population

In [19]:
df_q7.sort_values('violent_crime', ascending=False).head(1)

,state.area,state.division,Population,Income,Illiteracy,Life.Exp,Murder_x,HS.Grad,Frost,Area,Murder_y,Assault,UrbanPop,Rape,violent_crime
state.region,,,,,,,,,,,,,,,
2,56222.25,3.75,4208.125,4011.9375,1.7375,69.70625,10.58125,44.34375,64.625,54605.125,11.70625,220.0,59.4375,21.1625,252.86875


Region 2 had the most violent crime

## 8. **What SQL statement would you write to return two columns denoting income and Illiteracy in your state data?**

```sql
SELECT Income, Illiteracy FROM df_q5_final;
```

## 9. **What SQL statement would you write to return two columns denoting income and Illiteracy in your state data and sort the data from the highest to lowest income values and limit the data to incomes at or higher than 5000?**

```sql
SELECT Income, Illiteracy FROM df_q5_final WHERE Income >= 5000 ORDER BY Income DESC;
```

(I'm not sure why the semicolon disappears when I run the cell above)

## 10. **Create a new data frame that includes two columns from your state data denoting state names and violent crimes. Spread the state names to 50 unique columns with a single row that includes the violent crime data per state. Print the first five columns of the new dataset.**

In [20]:
df_q10 = df_q5_final[['state.name', 'violent_crime']]
df_q10.head()

,state.name,violent_crime
0,Alabama,270.4
1,Alaska,317.5
2,Arizona,333.1
3,Arkansas,218.3
4,California,325.6


In [21]:
df_q10_final = df_q10.pivot_table(columns='state.name', values='violent_crime')
df_q10_final.iloc[:, :5]

state.name,Alabama,Alaska,Arizona,Arkansas,California
violent_crime,270.4,317.5,333.1,218.3,325.6


## 11. **Take the dataset from question 10 and use a function to return a dictionary with each key denoting a state and each value indicating the square root of the value for violent crimes.**

In [22]:
def df_to_sqrt_dict(df_in, row_in):
    sqrt_values = df_in.loc[row_in].apply(math.sqrt)
    tmp_dict = sqrt_values.to_dict()
    return tmp_dict

In [23]:
dict_q11 = df_to_sqrt_dict(df_q10_final, 'violent_crime')
print(dict_q11)

{'Alabama': 16.443843832875572, 'Alaska': 17.81852968120546, 'Arizona': 18.251027368342857, 'Arkansas': 14.774978849392646, 'California': 18.04438970982394, 'Colorado': 15.830350596243914, 'Connecticut': 11.153474794878948, 'Delaware': 16.11521020650987, 'Florida': 19.552493447128423, 'Georgia': 15.943650773897426, 'Hawaii': 8.455767262643882, 'Idaho': 11.696153213770756, 'Illinois': 16.834488409215172, 'Indiana': 11.882760622010357, 'Iowa': 8.336666000266533, 'Kansas': 11.789826122551595, 'Kentucky': 11.61895003862225, 'Louisiana': 16.929264603047585, 'Maine': 9.638464608017191, 'Maryland': 18.414668066516974, 'Massachusetts': 13.026895255585654, 'Michigan': 17.383900597967074, 'Minnesota': 9.465727652959385, 'Mississippi': 17.093858546273278, 'Missouri': 14.669696656713798, 'Montana': 11.46298390472568, 'Nebraska': 11.081516141756055, 'Nevada': 17.61249556422939, 'New Hampshire': 8.282511696339462, 'New Jersey': 13.608820668963201, 'New Mexico': 18.12456896039186, 'New York': 17.0645

## 12. **Subset the list you created in question 12 to extract values for Texas and New York.**

In [24]:
subset_states = ['Texas', 'New York']
subset = {state: dict_q11[state] for state in subset_states}
print(subset)

{'Texas': 15.466091943344964, 'New York': 17.064583206161235}
